# FRP — Unsupervised Learning / Clustering
## Reproducible analysis of the frozen validated dataset

**Dataset:** `Dataset_v1_FROZEN` (53 specimens)

### Objective
Identify natural groupings in the physical/design characteristics of FRP-strengthened specimens **without using experimental outcomes to create the clusters**.

### Pre-specified feature sets
- **X1_full (17 variables):** complete physical/design set.
- **X2_reduced (13 variables):** reduced set defined before modeling after the correlation/multicollinearity analysis.

### Variables excluded from cluster formation
- `FM` — experimental failure mode
- `Vf_kN` — experimental FRP shear contribution
- `V_kN` — total experimental capacity
- `paper_ref` — source study/group
- identifiers and audit variables

`WC` remains in X1/X2 because it is a **pre-test physical FRP configuration**. It may be described by cluster, but it is not an independent external validation variable because it participates in cluster formation.

### Algorithms
1. K-Means
2. Agglomerative hierarchical clustering (Ward)
3. Gaussian Mixture Model (GMM)

### Candidate number of clusters
`k = 2, 3, 4, 5, 6`

### Primary internal validation
- Silhouette coefficient: higher is better
- Davies–Bouldin index: lower is better
- Calinski–Harabasz index: higher is better
- Cluster-size balance / minimum cluster size
- Stability by repeated 80% subsampling using Adjusted Rand Index (ARI)

### Selection rule
The primary solution is selected **without using FM, Vf, V or paper_ref**. Internal quality and stability are considered jointly; no single metric determines the winner.

After the solution is frozen, `FM`, `paper_ref`, `Vf_kN` and other descriptors are used only for post-hoc interpretation.

## 1. Install dependencies

In [ ]:
%pip -q install openpyxl scikit-learn scipy matplotlib

## 2. Imports and reproducibility configuration

In [ ]:
import os
import sys
import json
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal
from sklearn import __version__ as sklearn_version
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 20261001
K_VALUES = list(range(2, 7))
N_STABILITY_REPEATS = 100
SUBSAMPLE_FRACTION = 0.80

np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn_version)
print("Random state:", RANDOM_STATE)
print("Candidate k:", K_VALUES)

## 3. Upload/load the frozen workbook

In Google Colab, if the workbook is not already in `/content`, the cell opens the upload button automatically.

In [ ]:
EXPECTED_NAMES = [
    "FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx",
    "FRP_Dataset_v1_FINAL_VALIDATED_LOCKED(1).xlsx",
]

def locate_workbook():
    for name in EXPECTED_NAMES:
        p = Path("/content") / name
        if p.exists():
            return p
    # Also search current directory, useful outside Colab.
    for name in EXPECTED_NAMES:
        p = Path(name)
        if p.exists():
            return p
    return None

workbook = locate_workbook()

if workbook is None:
    try:
        from google.colab import files
        print("Workbook not found. Select the frozen validated Excel file.")
        uploaded = files.upload()
        workbook = locate_workbook()
    except Exception as exc:
        raise FileNotFoundError(
            "No workbook found. Upload FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx "
            "or FRP_Dataset_v1_FINAL_VALIDATED_LOCKED(1).xlsx."
        ) from exc

if workbook is None:
    raise FileNotFoundError("No workbook uploaded.")

df = pd.read_excel(workbook, sheet_name="Dataset_v1_FROZEN")

print("Loaded:", workbook.name)
print("Shape:", df.shape)

## 4. Dataset integrity checks

In [ ]:
required_columns = {
    "paper_ref", "specimen", "FM", "V_kN", "Vf_kN",
    "bw_mm", "d_mm", "df_mm", "a_d", "fc_MPa", "fsy_MPa",
    "rho_s", "fvy_MPa", "rho_sv", "tf_mm", "wf_mm", "sf_mm",
    "ffu_MPa", "Ef_GPa", "rho_fv", "WC", "alpha_f_deg"
}

missing_cols = required_columns - set(df.columns)

assert df.shape == (53, 32), f"Expected (53, 32), got {df.shape}"
assert not missing_cols, f"Missing columns: {sorted(missing_cols)}"
assert df.isna().sum().sum() == 0, "Dataset contains missing values."
assert df["paper_ref"].nunique() == 8, "Expected 8 studies."
assert set(df["FM"].unique()) == {"D", "R"}, "Expected FM classes D/R."

print("✓ Shape confirmed: 53 x 32")
print("✓ Missing values: 0")
print("✓ Studies:", df["paper_ref"].nunique())
print("✓ FM distribution:")
print(df["FM"].value_counts())

## 5. Freeze X1 and X2 exactly as defined before clustering

In [ ]:
features_X1 = [
    "bw_mm", "d_mm", "df_mm", "a_d", "fc_MPa", "fsy_MPa", "rho_s",
    "fvy_MPa", "rho_sv", "tf_mm", "wf_mm", "sf_mm", "ffu_MPa",
    "Ef_GPa", "rho_fv", "alpha_f_deg", "WC"
]

features_X2 = [
    "bw_mm", "df_mm", "a_d", "fc_MPa", "fsy_MPa", "rho_s",
    "fvy_MPa", "rho_sv", "ffu_MPa", "Ef_GPa", "rho_fv",
    "alpha_f_deg", "WC"
]

feature_sets = {
    "X1_full": features_X1,
    "X2_reduced": features_X2,
}

for name, cols in feature_sets.items():
    print(name, len(cols), "variables")
    print(cols)
    print()

print("Excluded from X2:", sorted(set(features_X1) - set(features_X2)))

# Explicit leakage guard
for forbidden in ["FM", "Vf_kN", "V_kN", "paper_ref", "specimen", "row_id"]:
    assert forbidden not in features_X1
    assert forbidden not in features_X2

print("✓ Outcome, study and identifier variables are excluded from cluster formation.")

## 6. Preprocessing

All numeric variables are standardized because distance-based clustering is scale-sensitive. `WC` is one-hot encoded.  
The transformer is fitted to the complete unsupervised feature matrix because there is no train/test target-learning step here. Stability analysis refits the clustering algorithms on subsamples.

In [ ]:
def make_preprocessor(features):
    categorical = ["WC"]
    numeric = [c for c in features if c not in categorical]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

transformed = {}
preprocessors = {}

for fs_name, features in feature_sets.items():
    prep = make_preprocessor(features)
    Z = prep.fit_transform(df[features])
    transformed[fs_name] = np.asarray(Z, dtype=float)
    preprocessors[fs_name] = prep
    print(fs_name, "raw:", df[features].shape, "transformed:", transformed[fs_name].shape)

print("✓ Preprocessing complete.")

## 7. Clustering functions and internal validation

In [ ]:
def fit_clusterer(algorithm, k, X, seed=RANDOM_STATE):
    if algorithm == "KMeans":
        model = KMeans(n_clusters=k, n_init=50, random_state=seed)
        labels = model.fit_predict(X)
    elif algorithm == "Agglomerative_Ward":
        model = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = model.fit_predict(X)
    elif algorithm == "GMM":
        model = GaussianMixture(
            n_components=k,
            covariance_type="full",
            n_init=20,
            reg_covar=1e-6,
            random_state=seed,
        )
        labels = model.fit_predict(X)
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    return model, np.asarray(labels)

def internal_metrics(X, labels):
    counts = pd.Series(labels).value_counts()
    return {
        "silhouette": silhouette_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
        "n_clusters_observed": int(counts.size),
    }

algorithms = ["KMeans", "Agglomerative_Ward", "GMM"]
print("Algorithms:", algorithms)

## 8. Stability analysis

For each full-data solution, 100 deterministic 80% subsamples are drawn. The same algorithm and `k` are refitted on each subsample. Stability is the ARI between the subsample clustering and the labels assigned by the full-data solution to those same specimens.

This is an internal robustness diagnostic; it does not use `FM`, `Vf_kN`, `V_kN` or `paper_ref`.

In [ ]:
def stability_against_full(X, algorithm, k, full_labels,
                           repeats=N_STABILITY_REPEATS,
                           fraction=SUBSAMPLE_FRACTION,
                           seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(X)
    m = max(k + 2, int(round(n * fraction)))
    aris = []

    for r in range(repeats):
        idx = np.sort(rng.choice(n, size=m, replace=False))
        X_sub = X[idx]

        try:
            _, labels_sub = fit_clusterer(
                algorithm, k, X_sub, seed=seed + r + 1
            )
            if len(np.unique(labels_sub)) < 2:
                continue
            aris.append(adjusted_rand_score(full_labels[idx], labels_sub))
        except Exception:
            continue

    if not aris:
        return np.nan, np.nan, 0

    return float(np.mean(aris)), float(np.std(aris, ddof=1)), len(aris)

print("✓ Stability function ready.")

## 9. Execute the complete unsupervised experiment

In [ ]:
evaluation_rows = []
assignment_rows = []

for fs_name, X in transformed.items():
    for algorithm in algorithms:
        for k in K_VALUES:
            print(f"[{fs_name}] {algorithm} k={k}")
            model, labels = fit_clusterer(algorithm, k, X)
            metrics = internal_metrics(X, labels)
            stab_mean, stab_sd, stab_n = stability_against_full(
                X, algorithm, k, labels
            )

            evaluation_rows.append({
                "feature_set": fs_name,
                "algorithm": algorithm,
                "k": k,
                **metrics,
                "stability_ARI_mean": stab_mean,
                "stability_ARI_sd": stab_sd,
                "stability_valid_repeats": stab_n,
            })

            for i, lab in enumerate(labels):
                assignment_rows.append({
                    "feature_set": fs_name,
                    "algorithm": algorithm,
                    "k": k,
                    "row_index": i,
                    "row_id": df.iloc[i]["row_id"],
                    "paper_ref": df.iloc[i]["paper_ref"],
                    "specimen": df.iloc[i]["specimen"],
                    "cluster": int(lab),
                })

evaluation = pd.DataFrame(evaluation_rows)
all_assignments = pd.DataFrame(assignment_rows)

print("\nSolutions evaluated:", len(evaluation))
print("Expected:", 2 * 3 * len(K_VALUES))
print("Assignment rows:", len(all_assignments))

assert len(evaluation) == 30
assert len(all_assignments) == 30 * 53

display(
    evaluation.sort_values(
        ["silhouette", "stability_ARI_mean"],
        ascending=[False, False]
    ).head(15)
)

## 10. Multi-criteria ranking and primary solution

To avoid selecting a solution using a single metric, each solution receives ranks for:
- Silhouette (higher better)
- Davies–Bouldin (lower better)
- Calinski–Harabasz (higher better)
- Stability ARI (higher better)

The mean of the four ranks is the pre-specified aggregate score. Solutions with a cluster containing fewer than 3 specimens are flagged as potentially fragile.

**Important:** FM, Vf, V and paper_ref are not used in this ranking.

In [ ]:
ranked = evaluation.copy()

ranked["rank_silhouette"] = ranked["silhouette"].rank(ascending=False, method="min")
ranked["rank_davies_bouldin"] = ranked["davies_bouldin"].rank(ascending=True, method="min")
ranked["rank_calinski_harabasz"] = ranked["calinski_harabasz"].rank(ascending=False, method="min")
ranked["rank_stability"] = ranked["stability_ARI_mean"].rank(ascending=False, method="min")

ranked["aggregate_rank"] = ranked[
    ["rank_silhouette", "rank_davies_bouldin",
     "rank_calinski_harabasz", "rank_stability"]
].mean(axis=1)

ranked["fragile_small_cluster"] = ranked["min_cluster_size"] < 3

eligible = ranked[~ranked["fragile_small_cluster"]].copy()
if eligible.empty:
    eligible = ranked.copy()

eligible = eligible.sort_values(
    ["aggregate_rank", "silhouette", "stability_ARI_mean"],
    ascending=[True, False, False]
).reset_index(drop=True)

primary = eligible.iloc[0]

PRIMARY_FEATURE_SET = primary["feature_set"]
PRIMARY_ALGORITHM = primary["algorithm"]
PRIMARY_K = int(primary["k"])

print("=== PRIMARY INTERNAL SOLUTION ===")
print("Feature set:", PRIMARY_FEATURE_SET)
print("Algorithm:", PRIMARY_ALGORITHM)
print("k:", PRIMARY_K)
print("Silhouette:", round(primary["silhouette"], 4))
print("Davies-Bouldin:", round(primary["davies_bouldin"], 4))
print("Calinski-Harabasz:", round(primary["calinski_harabasz"], 4))
print("Stability ARI:", round(primary["stability_ARI_mean"], 4))
print("Minimum cluster size:", int(primary["min_cluster_size"]))

display(eligible.head(10))

## 11. Freeze primary cluster assignments

Cluster numbers are arbitrary labels; `0`, `1`, etc. do not imply ordering.

In [ ]:
X_primary = transformed[PRIMARY_FEATURE_SET]
primary_model, primary_labels = fit_clusterer(
    PRIMARY_ALGORITHM, PRIMARY_K, X_primary
)

primary_assignments = df[
    ["row_id", "paper_ref", "specimen", "FM", "WC", "V_kN", "Vf_kN"]
].copy()
primary_assignments["cluster"] = primary_labels

print("Cluster sizes:")
print(primary_assignments["cluster"].value_counts().sort_index())
display(primary_assignments.head())

## 12. PCA visualization of the primary solution

PCA is used **only for visualization**, not to create the primary clusters.

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_primary)

pca_df = primary_assignments.copy()
pca_df["PC1"] = coords[:, 0]
pca_df["PC2"] = coords[:, 1]

print(
    "Explained variance PC1+PC2:",
    round(float(pca.explained_variance_ratio_[:2].sum()), 4)
)

plt.figure(figsize=(8, 6))
for c in sorted(pca_df["cluster"].unique()):
    g = pca_df[pca_df["cluster"] == c]
    plt.scatter(g["PC1"], g["PC2"], label=f"Cluster {c}", s=55, alpha=0.8)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(
    f"PCA visualization — {PRIMARY_FEATURE_SET}, "
    f"{PRIMARY_ALGORITHM}, k={PRIMARY_K}"
)
plt.legend()
plt.tight_layout()
plt.show()

## 13. Post-hoc interpretation — performed only after the primary solution is frozen

These variables **do not participate in selecting the clustering solution**:
- `FM`
- `paper_ref`
- `Vf_kN`
- `V_kN`

`WC` is also tabulated, but remember that it is part of X1/X2 and therefore is descriptive rather than independent validation.

In [ ]:
print("=== Cluster x FM ===")
cluster_fm = pd.crosstab(
    primary_assignments["cluster"],
    primary_assignments["FM"],
    margins=True
)
display(cluster_fm)

print("=== Cluster x study (paper_ref) ===")
cluster_study = pd.crosstab(
    primary_assignments["cluster"],
    primary_assignments["paper_ref"],
    margins=True
)
display(cluster_study)

print("=== Cluster x WC ===")
cluster_wc = pd.crosstab(
    primary_assignments["cluster"],
    primary_assignments["WC"],
    margins=True
)
display(cluster_wc)

print("=== Vf_kN by cluster ===")
vf_by_cluster = primary_assignments.groupby("cluster")["Vf_kN"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
display(vf_by_cluster)

# Descriptive external association indices
fm_codes = pd.Categorical(primary_assignments["FM"]).codes
study_codes = pd.Categorical(primary_assignments["paper_ref"]).codes

print("NMI(cluster, FM):",
      round(normalized_mutual_info_score(primary_labels, fm_codes), 4))
print("ARI(cluster, FM):",
      round(adjusted_rand_score(primary_labels, fm_codes), 4))
print("NMI(cluster, study):",
      round(normalized_mutual_info_score(primary_labels, study_codes), 4))
print("ARI(cluster, study):",
      round(adjusted_rand_score(primary_labels, study_codes), 4))

groups_vf = [
    g["Vf_kN"].to_numpy()
    for _, g in primary_assignments.groupby("cluster")
]
if len(groups_vf) >= 2 and all(len(g) > 0 for g in groups_vf):
    H, p = kruskal(*groups_vf)
    print("Kruskal-Wallis Vf_kN across clusters: H =", round(H, 4),
          "p =", round(p, 6))

## 14. Cluster profiles in original physical units

In [ ]:
profile_features = [c for c in feature_sets[PRIMARY_FEATURE_SET] if c != "WC"]

profiles = df[profile_features].copy()
profiles["cluster"] = primary_labels

cluster_profiles_mean = profiles.groupby("cluster").mean(numeric_only=True)
cluster_profiles_median = profiles.groupby("cluster").median(numeric_only=True)

print("=== Mean physical profile ===")
display(cluster_profiles_mean)

print("=== Median physical profile ===")
display(cluster_profiles_median)

## 15. Export all reproducibility outputs

In [ ]:
OUTPUT_DIR = Path("/content/clustering_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

evaluation.to_csv(OUTPUT_DIR / "clustering_all_solutions.csv", index=False)
ranked.to_csv(OUTPUT_DIR / "clustering_ranked_solutions.csv", index=False)
all_assignments.to_csv(OUTPUT_DIR / "clustering_all_assignments.csv", index=False)
primary_assignments.to_csv(OUTPUT_DIR / "clustering_primary_assignments.csv", index=False)
pca_df.to_csv(OUTPUT_DIR / "clustering_primary_pca.csv", index=False)
cluster_profiles_mean.to_csv(OUTPUT_DIR / "cluster_profiles_mean.csv")
cluster_profiles_median.to_csv(OUTPUT_DIR / "cluster_profiles_median.csv")
cluster_fm.to_csv(OUTPUT_DIR / "cluster_vs_FM.csv")
cluster_study.to_csv(OUTPUT_DIR / "cluster_vs_study.csv")
cluster_wc.to_csv(OUTPUT_DIR / "cluster_vs_WC.csv")
vf_by_cluster.to_csv(OUTPUT_DIR / "Vf_by_cluster.csv")

primary_record = {
    "feature_set": PRIMARY_FEATURE_SET,
    "algorithm": PRIMARY_ALGORITHM,
    "k": PRIMARY_K,
    "selection_basis": (
        "Mean rank of Silhouette, Davies-Bouldin, Calinski-Harabasz "
        "and subsampling stability ARI; solutions with min cluster size <3 "
        "flagged as fragile. FM/Vf/V/paper_ref excluded from selection."
    ),
    "random_state": RANDOM_STATE,
    "k_values": K_VALUES,
    "stability_repeats": N_STABILITY_REPEATS,
    "subsample_fraction": SUBSAMPLE_FRACTION,
}

with open(OUTPUT_DIR / "primary_solution.json", "w", encoding="utf-8") as f:
    json.dump(primary_record, f, indent=2)

versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn_version,
    "random_state": RANDOM_STATE,
}
with open(OUTPUT_DIR / "environment_versions.json", "w", encoding="utf-8") as f:
    json.dump(versions, f, indent=2)

# Save PCA figure
plt.figure(figsize=(8, 6))
for c in sorted(pca_df["cluster"].unique()):
    g = pca_df[pca_df["cluster"] == c]
    plt.scatter(g["PC1"], g["PC2"], label=f"Cluster {c}", s=55, alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(
    f"PCA visualization — {PRIMARY_FEATURE_SET}, "
    f"{PRIMARY_ALGORITHM}, k={PRIMARY_K}"
)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "primary_clustering_PCA.png", dpi=300, bbox_inches="tight")
plt.close()

print("All outputs written to:", OUTPUT_DIR)
print("Files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

## 16. Create a ZIP backup before leaving Colab

`/content` is temporary. Download the ZIP immediately after it is created.

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/FRP_clustering_results",
    "zip",
    "/content/clustering_results"
)

print("Created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Automatic download not available. Download manually from the Files panel.")

# Interpretation guardrails

1. Do **not** call clusters “failure-mode clusters” merely because their FM proportions differ.
2. Do **not** use post-hoc agreement with FM or study to choose `k`.
3. A high association with `paper_ref` may indicate **study/domain structure**, not necessarily a universal physical taxonomy.
4. Cluster labels are arbitrary.
5. With N=53, stability and physical interpretability matter more than small differences in any single internal index.
6. The primary solution should be interpreted together with alternative high-ranking solutions, especially if their aggregate ranks are close.